# Apartment Data Preprocessing

This notebook preprocesses the cleaned apartment dataset and exports model-ready features.

## Flow
1. Setup and load cleaned apartment data
2. Parse address into structured location parts
3. Build preprocessing pipeline (Yes/No binary + OHE + frequency + numeric passthrough)
4. Export preprocessed matrix
5. Export model feature block and OOF location encoded features

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('Environment setup complete.')

Environment setup complete.


## Step 2 - Workflow Cell 2
Load input data and inspect basic structure for this step.

In [2]:
df = pd.read_csv('../../data/apartment/satilir_properties_apartment_cleaned.csv')
print('Dataset shape:', df.shape)
print('Total missing values:', int(df.isna().sum().sum()))

df.head()

Dataset shape: (4273, 26)
Total missing values: 0


,price,rooms,area_m2,floor,has_document,address,avtodayanacaq,balkon,duzelme,esyali,hovuz,internet,isiq,kabel_tv,kombi,kondisioner,lift,merkezi_qizdirici_sistem,metbex_mebeli,pvc_pencere,qaz,su,telefon,temirli,rooms_group,rooms_group_code
0,200000,3.0,65.0,5.0,No,"Bakı, Nərimanov, metro 28 May",Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,No,Yes,3-4,2
1,800000,4.0,197.0,9.0,Yes,"Bakı, Nərimanov, metro Gənclik",Yes,Yes,No,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,3-4,2
2,210000,2.0,54.0,12.0,No,"Bakı, Nərimanov, metro Nəriman Nərimanov",No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,1-2,1
3,149000,4.0,100.0,2.0,Yes,"Bakı, Binəqədi, Biləcəri qəs.",No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,3-4,2
4,45000,1.0,44.0,1.0,Yes,Xırdalan,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,1-2,1


## Step 3 - Workflow Cell 3
Run this notebook step and prepare outputs for the next stage.

In [3]:
yes_no_cols = []
for col in df.columns:
    non_null = df[col].dropna().astype(str).str.strip().str.lower()
    if len(non_null) and set(non_null.unique()).issubset({"yes", "no"}):
        yes_no_cols.append(col)

print("Detected Yes/No columns:", yes_no_cols)

Detected Yes/No columns: ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirli']


## Step 4 - Workflow Cell 4
Run this notebook step and prepare outputs for the next stage.

In [4]:
print(f"Before separating address: {df.shape[1]} columns")

Before separating address: 26 columns


## Step 5 - Workflow Cell 5
Run this notebook step and prepare outputs for the next stage.

In [5]:
candidate_cols = ["address"]
address_col = next((c for c in candidate_cols if c in df.columns), None)

if address_col is None:
    raise ValueError(f"No address column found. Available columns: {list(df.columns)}")

address_parts = ["address_part_1", "address_part_2", "address_part_3", "address_part_4"]

parts = (
    df["address"]
    .fillna("")
    .astype(str)
    .str.split(",", expand=True)
    .iloc[:, :4]
)
parts.columns = address_parts[:parts.shape[1]]

for col in address_parts:
    if col not in parts.columns:
        parts[col] = ""


df = pd.concat([df, parts[address_parts]], axis=1)

df[address_parts].head()

,address_part_1,address_part_2,address_part_3,address_part_4
0,Bakı,Nərimanov,metro 28 May,None
1,Bakı,Nərimanov,metro Gənclik,None
2,Bakı,Nərimanov,metro Nəriman Nərimanov,None
3,Bakı,Binəqədi,Biləcəri qəs.,None
4,Xırdalan,None,None,None


## Step 6 - Workflow Cell 6
Run this notebook step and prepare outputs for the next stage.

In [6]:
print(f"After separating address: {df.shape[1]} columns")

After separating address: 30 columns


## Step 7 - Workflow Cell 7
Run this notebook step and prepare outputs for the next stage.

In [7]:
df["address"]

0                         Bakı, Nərimanov, metro 28 May
1                        Bakı, Nərimanov, metro Gənclik
2              Bakı, Nərimanov, metro Nəriman Nərimanov
3                         Bakı, Binəqədi, Biləcəri qəs.
4                                              Xırdalan
                             ...                       
4268                             Bakı, Abşeron, Masazır
4269                                           Xırdalan
4270    Bakı, Binəqədi, Biləcəri qəs., metro Avtovağzal
4271                   Bakı, Yasamal, metro İnşaatçılar
4272                    Bakı, Xətai, metro Həzi Aslanov
Name: address, Length: 4273, dtype: object

## Step 8 - Workflow Cell 8
Run this notebook step and prepare outputs for the next stage.

In [8]:
df.drop(columns=["address"], inplace=True)

## Step 9 - Workflow Cell 9
Run this notebook step and prepare outputs for the next stage.

In [9]:
df[address_parts].isnull().sum()/len(df)

address_part_1    0.000000
address_part_2    0.138076
address_part_3    0.199860
address_part_4    0.760590
dtype: float64

## Step 10 - Workflow Cell 10
Run this notebook step and prepare outputs for the next stage.

In [10]:
df["address_part_4"].unique()

array([None, ' metro Həzi Aslanov', ' metro Avtovağzal',
       ' metro Neftçilər', ' metro Əhmədli', ' metro İnşaatçılar',
       ' metro İçərişəhər', ' metro Nəsimi', ' metro Azadlıq',
       ' metro Elmlər akademiyası', ' metro 20 Yanvar', ' metro Dərnəgül',
       ' metro Xalqlar dostluğu', ' metro Gənclik', ' metro Qara Qarayev',
       ' metro Koroğlu', ' metro Xətai', ' metro Memar Əcəmi',
       ' metro Nəriman Nərimanov', ' metro Memar Əcəmi - 2',
       ' metro 8 Noyabr', ' metro 28 May', ' metro Nizami',
       ' metro Xocaəsən', ' metro Sahil'], dtype=object)

## Step 11 - Workflow Cell 11
Run this notebook step and prepare outputs for the next stage.

In [11]:
df.drop(columns=["address_part_4"], inplace=True)

## Step 12 - Workflow Cell 12
Run this notebook step and prepare outputs for the next stage.

In [12]:
df["address_part_3"].unique()

array([' metro 28 May', ' metro Gənclik', ' metro Nəriman Nərimanov',
       ' Biləcəri qəs.', None, ' metro Qara Qarayev', ' Əhmədli',
       ' Yeni Günəşli qəs.', ' metro İnşaatçılar', ' Masazır', ' Buzovna',
       ' 8-ci kilometr', ' Yasamal qəs.', ' metro Azadlıq',
       ' Yeni Yasamal qəs.', ' 9-cu mikrorayon', ' metro Koroğlu',
       ' metro Neftçilər', ' metro 20 Yanvar', ' H.Aslanov qəs.',
       ' metro Memar Əcəmi', ' metro İçərişəhər', ' 8-ci mikrorayon',
       ' metro Həzi Aslanov', ' metro Xalqlar dostluğu', ' metro Xətai',
       ' metro Elmlər akademiyası', ' 7-ci mikrorayon', ' metro 8 Noyabr',
       ' Saray', ' metro Nizami', ' Lökbatan qəs.', ' Badamdar qəs.',
       ' Qara şəhər', ' metro Əhmədli', ' metro Sahil', ' Nardaran qəs.',
       ' 3-cü mikrorayon', ' Köhnə Günəşli qəs.', ' metro Nəsimi',
       ' Əmircan qəs.', ' Bakıxanov qəs.', ' Qaraçuxur qəs.',
       ' Hövsan qəs.', ' Ramana qəs.', ' Ağ şəhər', ' 6-cı mikrorayon',
       ' Bayıl qəs.', ' Montin qə

## Step 13 - Workflow Cell 13
Run this notebook step and prepare outputs for the next stage.

In [13]:
df.drop(columns=["address_part_3"], inplace=True)

## Step 14 - Workflow Cell 14
Run this notebook step and prepare outputs for the next stage.

In [14]:
df["address_part_2"].unique()

array([' Nərimanov', ' Binəqədi', None, ' Nəsimi', ' Nizami', ' Xətai',
       ' Suraxanı', ' Yasamal', ' Abşeron', ' Xəzər', ' Sabunçu',
       ' Səbail', ' 10-cu mikrorayon', ' 1-ci mikrorayon', ' Qaradağ',
       ' St.Sumqayıt', ' 18-ci mikrorayon', ' 44-cü məhəllə',
       ' Sumqayıt Bulvarı', ' 14-cü məhəllə', ' 9-cu mikrorayon',
       ' 6-ci mikrorayon', ' 17-ci mikrorayon', ' 16-ci mikrorayon',
       ' 8-ci mikrorayon', ' 12-ci mikrorayon', ' 13-cü mikrorayon',
       ' 1-ci məhəllə', ' metro Memar Əcəmi', ' 36-ci məhəllə',
       ' 2-ci mikrorayon', ' 3-ci məhəllə', ' 15-ci məhəllə',
       ' 40-ci məhəllə'], dtype=object)

## Step 15 - Workflow Cell 15
Run this notebook step and prepare outputs for the next stage.

In [15]:
df["address_part_1"].unique()

array(['Bakı', 'Xırdalan', 'Lənkəran', 'Sumqayıt', 'Qəbələ', 'Gəncə'],
      dtype=object)

## Step 16 - Workflow Cell 16
Run this notebook step and prepare outputs for the next stage.

In [16]:
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(f"Total columns: {df.shape[1]}")
print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Yes No columns ({len(yes_no_cols)}): {yes_no_cols}")
print(f"Address parts (2): address_part_1, address_part_2")

schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
})
schema_df

Total columns: 27
Numeric (5): ['price', 'rooms', 'area_m2', 'floor', 'rooms_group_code']
Categorical (22): ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirli', 'rooms_group', 'address_part_1', 'address_part_2']
Yes No columns (19): ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirli']
Address parts (2): address_part_1, address_part_2


,column,dtype
0,price,int64
1,rooms,float64
2,area_m2,float64
3,floor,float64
4,has_document,object
5,avtodayanacaq,object
6,balkon,object
7,duzelme,object
8,esyali,object
9,hovuz,object


## Step 17 - Workflow Cell 17
Run this notebook step and prepare outputs for the next stage.

In [17]:
class YesNoBinaryEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.feature_names_in_ = X_df.columns.to_numpy()
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.feature_names_in_)
        out = pd.DataFrame(index=X_df.index)
        for col in self.feature_names_in_:
            out[col] = (
                X_df[col]
                .astype('string')
                .str.strip()
                .str.lower()
                .map({'yes': 1, 'no': 0})
                .astype(float)
            )
        return out.to_numpy()

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return np.asarray(input_features, dtype=object)


class MultiColumnFrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps_ = {}
        self.columns_ = []

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = X_df.columns.tolist()
        self.feature_names_in_ = np.asarray(self.columns_, dtype=object)
        self.freq_maps_ = {
            col: X_df[col].astype('string').value_counts(normalize=True, dropna=False).to_dict()
            for col in self.columns_
        }
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.columns_)
        out = pd.DataFrame(index=X_df.index)
        for col in self.columns_:
            out[f'{col}_freq'] = (
                X_df[col].astype('string').map(self.freq_maps_[col]).fillna(0.0).astype(float)
            )
        return out.to_numpy()

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)
        return np.asarray([f'{col}_freq' for col in input_features], dtype=object)


y = df["price"].copy()
X_df = df.drop(columns=["price"], errors="ignore").copy()

yes_no_cols_model = [c for c in yes_no_cols if c in X_df.columns]

categorical_cols = X_df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
categorical_base = [c for c in categorical_cols if c not in yes_no_cols_model]


cardinality = X_df[categorical_base].nunique(dropna=False).sort_values(ascending=False)
cardinality_cutoff = 49 # 1% of data
ohe_card_cols = [c for c in categorical_base if X_df[c].nunique(dropna=False) <= cardinality_cutoff]
high_card_cols = [c for c in categorical_base if X_df[c].nunique(dropna=False) > cardinality_cutoff]

numeric_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()

print("Yes/No columns:", yes_no_cols_model)
print(f"OHE categorical (<= {cardinality_cutoff}):", ohe_card_cols)
print(f"High-cardinality categorical (> {cardinality_cutoff}):", high_card_cols)
print("Numeric columns:", len(numeric_cols))
print("Top categorical cardinalities:")
print(cardinality.head(20))

try:
    ohe_cat = OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=0.01,
        sparse_output=False,
    )
except TypeError:
    ohe_cat = OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("yes_no_binary", Pipeline([("yes_no", YesNoBinaryEncoder())]), yes_no_cols_model),
        ("cat_ohe", Pipeline([("ohe", ohe_cat)]), ohe_card_cols),
        ("cat_freq_high", Pipeline([("freq", MultiColumnFrequencyEncoder())]), high_card_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
    sparse_threshold=0,
)

full_pipeline = Pipeline([("preprocessor", preprocessor)])

X_processed = full_pipeline.fit_transform(X_df)
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
processed_df = pd.DataFrame(X_processed, columns=feature_names, index=X_df.index)

print("Processed feature matrix shape:", processed_df.shape)
processed_df.head()

Yes/No columns: ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirli']
OHE categorical (<= 49): ['rooms_group', 'address_part_1', 'address_part_2']
High-cardinality categorical (> 49): []
Numeric columns: 4
Top categorical cardinalities:
address_part_2    34
address_part_1     6
rooms_group        4
dtype: int64
Processed feature matrix shape: (4273, 43)


,yes_no_binary__has_document,yes_no_binary__avtodayanacaq,yes_no_binary__balkon,yes_no_binary__duzelme,yes_no_binary__esyali,yes_no_binary__hovuz,yes_no_binary__internet,yes_no_binary__isiq,yes_no_binary__kabel_tv,yes_no_binary__kombi,yes_no_binary__kondisioner,yes_no_binary__lift,yes_no_binary__merkezi_qizdirici_sistem,yes_no_binary__metbex_mebeli,yes_no_binary__pvc_pencere,yes_no_binary__qaz,yes_no_binary__su,yes_no_binary__telefon,yes_no_binary__temirli,cat_ohe__rooms_group_1-2,cat_ohe__rooms_group_3-4,cat_ohe__rooms_group_5-6,cat_ohe__rooms_group_infrequent_sklearn,cat_ohe__address_part_1_Bakı,cat_ohe__address_part_1_Sumqayıt,cat_ohe__address_part_1_Xırdalan,cat_ohe__address_part_1_infrequent_sklearn,cat_ohe__address_part_2_ Abşeron,cat_ohe__address_part_2_ Binəqədi,cat_ohe__address_part_2_ Nizami,cat_ohe__address_part_2_ Nərimanov,cat_ohe__address_part_2_ Nəsimi,cat_ohe__address_part_2_ Sabunçu,cat_ohe__address_part_2_ Suraxanı,cat_ohe__address_part_2_ Səbail,cat_ohe__address_part_2_ Xətai,cat_ohe__address_part_2_ Yasamal,cat_ohe__address_part_2_None,cat_ohe__address_part_2_infrequent_sklearn,num__rooms,num__area_m2,num__floor,num__rooms_group_code
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,65.0,5.0,2.0
1,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,197.0,9.0,2.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,54.0,12.0,1.0
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,100.0,2.0,2.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,44.0,1.0,1.0


## Step 18 - Workflow Cell 18
Persist generated outputs and artifacts for downstream use.

In [18]:
processed_df.to_csv('../../data/apartment/satilir_properties_apartment_processed.csv', index=False)
print('Preprocessed dataset saved to', '../../data/apartment/satilir_properties_apartment_preprocessed.csv')
y.to_csv('../../data/apartment/satilir_properties_apartment_target.csv', index=False)
print('Target variable saved to', '../../data/apartment/satilir_properties_apartment_target.csv')

Preprocessed dataset saved to ../../data/apartment/satilir_properties_apartment_preprocessed.csv
Target variable saved to ../../data/apartment/satilir_properties_apartment_target.csv
